# 01 – Data Prep

**Goal:** turn MoneyPuck's raw 2025-26 skater file into one clean table (one row per player) that the WAR model can use.

What this notebook does:
1. Load the data and look at how it's structured
2. Reshape it so each player has separate columns for even strength (5v5), power play, and penalty kill
3. Calculate per-60 rates correctly (fixing the v1 bugs)
4. Define a real **replacement level**: what a fringe, call-up level player produces
5. Save the cleaned table for the next notebook

Data: MoneyPuck.com (season-level skaters, 2025-26).

## 1. Setup
`pandas` handles tables, `numpy` handles math. `DATA_PATH` uses `../` because this notebook lives in `notebooks/` and the data lives in `data/`.

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

DATA_PATH = "../data/skaters25-26.csv"

## 2. Load and inspect
Before doing anything, look at the data. Key things to notice:
- Each player appears **5 times**, once per game situation (`all`, `5on5`, `5on4`, `4on5`, `other`)
- `icetime` is in **seconds**, not minutes

In [2]:
raw = pd.read_csv(DATA_PATH).copy()   # .copy() keeps the table tidy in memory

print("Rows, columns:", raw.shape)
print("Unique players:", raw["playerId"].nunique())
print()
print(raw["situation"].value_counts())

raw[["name", "team", "position", "situation", "games_played", "icetime"]].head(10)

Rows, columns: (4700, 154)
Unique players: 940

situation
other    940
all      940
5on5     940
4on5     940
5on4     940
Name: count, dtype: int64


,name,team,position,situation,games_played,icetime
0,Sam Lafferty,CHI,C,other,29,85.0
1,Sam Lafferty,CHI,C,all,29,14959.0
2,Sam Lafferty,CHI,C,5on5,29,14856.0
3,Sam Lafferty,CHI,C,4on5,29,0.0
4,Sam Lafferty,CHI,C,5on4,29,18.0
5,Travis Mitchell,NYI,D,other,9,33.0
6,Travis Mitchell,NYI,D,all,9,6230.0
7,Travis Mitchell,NYI,D,5on5,9,6152.0
8,Travis Mitchell,NYI,D,4on5,9,45.0
9,Travis Mitchell,NYI,D,5on4,9,0.0


## 3. Position groups
Forwards (C, L, R) and defensemen (D) play very different roles, so they get separate replacement levels later. This adds a simple `F` / `D` label.

In [3]:
raw["pos_group"] = np.where(raw["position"] == "D", "D", "F")
raw.loc[raw["situation"] == "all", "pos_group"].value_counts()

pos_group
F    615
D    325
Name: count, dtype: int64

## 4. Reshape: one row per player
Right now a player is spread across 5 rows. The model needs one row per player, with stats split by situation:

| Prefix | Situation | Why it matters |
|---|---|---|
| `ev_` | 5-on-5 (even strength) | Most of the game; the core of player value |
| `pp_` | 5-on-4 (power play) | Separate skill, separate value |
| `pk_` | 4-on-5 (penalty kill) | Separate skill, separate value |
| `all_` | Everything combined | Used for total ice time, penalties, faceoffs |

`STAT_COLS` is the short list of stats we actually need. v1 used ~15 overlapping stats; v2 keeps only what the model uses.

In [4]:
STAT_COLS = [
    "icetime",
    "I_F_goals", "I_F_primaryAssists", "I_F_secondaryAssists",
    "I_F_xGoals",                                  # individual expected goals
    "OnIce_F_xGoals", "OnIce_A_xGoals",            # team xG for / against while player is on ice
    "onIce_xGoalsPercentage", "offIce_xGoalsPercentage",
    "penalties", "penaltiesDrawn",
    "faceoffsWon", "faceoffsLost",
]

SIT_MAP = {"all": "all", "5on5": "ev", "5on4": "pp", "4on5": "pk"}

sub = raw[raw["situation"].isin(SIT_MAP.keys())].copy()
sub["sit"] = sub["situation"].map(SIT_MAP)

wide = sub.pivot_table(index="playerId", columns="sit", values=STAT_COLS, aggfunc="first")
wide.columns = [f"{sit}_{stat}" for stat, sit in wide.columns]

info = (raw.loc[raw["situation"] == "all", ["playerId", "name", "team", "position", "pos_group", "games_played"]]
           .set_index("playerId"))

players = info.join(wide).reset_index()

# seconds -> minutes
for sit in SIT_MAP.values():
    players[f"{sit}_toi"] = players[f"{sit}_icetime"] / 60

print("Players:", len(players), "| Columns:", players.shape[1])
players[["name", "team", "pos_group", "games_played", "all_toi", "ev_toi", "pp_toi", "pk_toi"]].head()

Players: 940 | Columns: 62


,name,team,pos_group,games_played,all_toi,ev_toi,pp_toi,pk_toi
0,Sam Lafferty,CHI,F,29,249.316667,247.600000,0.300000,0.000000
1,Travis Mitchell,NYI,D,9,103.833333,102.533333,0.000000,0.750000
2,Erik Gudbranson,CBJ,D,37,657.533333,539.566667,0.033333,100.266667
3,Mason Marchment,CBJ,F,68,1181.283333,992.266667,152.700000,1.383333
4,Jacob Bryson,WPG,D,50,543.150000,525.750000,0.050000,13.583333


## 5. Per-60 rates (fixing v1)
A **per-60 rate** answers: "how much of this does a player produce per 60 minutes on ice?" It lets us compare a 22-minute defenseman to a 12-minute forward.

One function does it everywhere so the math is consistent. If a player has 0 minutes, the rate is blank (NaN) instead of crashing.

**v1 bugs fixed here:**
- v1 multiplied Corsi *percentage* by 60, which isn't a real number. Dropped.
- v1's "finishing" was really shot quality. Real finishing = **goals minus expected goals** (did he score more or less than his chances deserved?).
- v1 counted penalties twice. Now there's one penalty number: drawn minus taken.

In [5]:
def per60(count, toi_minutes):
    """Rate per 60 minutes; NaN when the player has no ice time."""
    return np.where(toi_minutes > 0, count / toi_minutes * 60, np.nan)

# Even strength (5v5)
players["ev_goals_60"]     = per60(players["ev_I_F_goals"], players["ev_toi"])
players["ev_a1_60"]        = per60(players["ev_I_F_primaryAssists"], players["ev_toi"])
players["ev_ixg_60"]       = per60(players["ev_I_F_xGoals"], players["ev_toi"])
players["ev_xgf_60"]       = per60(players["ev_OnIce_F_xGoals"], players["ev_toi"])
players["ev_xga_60"]       = per60(players["ev_OnIce_A_xGoals"], players["ev_toi"])

# Finishing: goals scored above/below what his shots were expected to produce
players["ev_finishing"]    = players["ev_I_F_goals"] - players["ev_I_F_xGoals"]

# Team-relative impact: on-ice xG% minus the team's xG% when he's on the bench.
# Positive = the team does better with him on the ice. This is how we reduce team-quality bias.
players["ev_xg_pct_rel"]   = players["ev_onIce_xGoalsPercentage"] - players["ev_offIce_xGoalsPercentage"]

# Penalties (all situations): drawn minus taken
players["pen_diff"]        = players["all_penaltiesDrawn"] - players["all_penalties"]

# Faceoffs (matters for centers)
fo_total = players["all_faceoffsWon"] + players["all_faceoffsLost"]
players["fo_pct"]          = np.where(fo_total >= 100, players["all_faceoffsWon"] / fo_total, np.nan)

players = players.copy()

players[["name", "pos_group", "ev_toi", "ev_goals_60", "ev_ixg_60", "ev_finishing", "ev_xg_pct_rel", "pen_diff"]].head()

,name,pos_group,ev_toi,ev_goals_60,ev_ixg_60,ev_finishing,ev_xg_pct_rel,pen_diff
0,Sam Lafferty,F,247.600000,0.242326,0.205977,0.15,-0.02,-1.0
1,Travis Mitchell,D,102.533333,0.585176,0.222367,0.62,-0.09,-1.0
2,Erik Gudbranson,D,539.566667,0.111200,0.115648,-0.04,-0.19,-3.0
3,Mason Marchment,F,992.266667,0.846547,0.724402,2.02,0.05,-6.0
4,Jacob Bryson,D,525.750000,0.228245,0.119829,0.95,0.01,-1.0


## 6. Sanity check
Always check that numbers pass the eye test before building on them. Top 10 even-strength expected-goal rates (min 500 EV minutes) should be mostly names you recognize as elite shooters.

In [6]:
qualified = players[players["ev_toi"] >= 500]

(qualified.sort_values("ev_ixg_60", ascending=False)
          [["name", "team", "position", "ev_toi", "ev_ixg_60", "ev_goals_60", "ev_finishing"]]
          .head(10)
          .round(2))

,name,team,position,ev_toi,ev_ixg_60,ev_goals_60,ev_finishing
31,Zach Hyman,EDM,L,893.58,1.23,0.94,-4.38
490,Anders Lee,NYI,L,1055.02,1.20,0.51,-12.03
928,Brady Tkachuk,OTT,L,759.17,1.19,0.87,-4.10
248,Artturi Lehkonen,COL,L,1013.75,1.11,1.07,-0.72
209,Brandon Hagel,TBL,L,1008.93,1.10,1.37,4.42
834,Auston Matthews,TOR,C,950.80,1.10,1.07,-0.49
411,Ross Colton,COL,C,843.23,1.10,0.50,-8.43
520,Timo Meier,NJD,R,1192.58,1.08,0.65,-8.55
404,Jason Zucker,BUF,L,743.38,1.07,1.13,0.76
398,Jackson Blake,CAR,R,1118.98,1.06,0.75,-5.82


## 7. Replacement level
v1 compared everyone to the average of *good* players, which made the baseline too high. WAR needs a **replacement level**: what a team gets from a fringe player it could call up for free.

**Definition used here:**
- On each team, rank forwards and defensemen by total ice time
- Top **13 forwards** and top **7 defensemen** = **regulars** (a full lineup plus one extra)
- Everyone else = the **replacement pool**

**Known limitation:** MoneyPuck lists a traded player under one team only, so a few depth rankings will be slightly off. Worth noting in the README.

In [7]:
players["toi_rank"] = (players.groupby(["team", "pos_group"])["all_toi"]
                              .rank(ascending=False, method="first"))

lineup_size = players["pos_group"].map({"F": 13, "D": 7})
players["is_regular"] = players["toi_rank"] <= lineup_size

pd.crosstab(players["pos_group"], players["is_regular"].map({True: "regular", False: "replacement"}))

is_regular,regular,replacement
pos_group,,
D,224,101
F,416,199


### Replacement rates
For the replacement pool, only players with **100+ even-strength minutes** count, so a guy who played 2 games doesn't skew things.

Rates are **pooled**: total goals ÷ total minutes for the whole group, not an average of each player's rate. Averaging rates would let tiny-sample players swing the result.

The table compares replacement players to regulars. Replacement should be clearly worse on offense; if not, something's wrong.

In [8]:
RATE_STATS = {
    "goals_60":  "ev_I_F_goals",
    "a1_60":     "ev_I_F_primaryAssists",
    "ixg_60":    "ev_I_F_xGoals",
    "xgf_60":    "ev_OnIce_F_xGoals",
    "xga_60":    "ev_OnIce_A_xGoals",
}

def pooled_rates(df):
    return pd.Series({name: df[col].sum() / df["ev_toi"].sum() * 60 for name, col in RATE_STATS.items()})

pool = players[(~players["is_regular"]) & (players["ev_toi"] >= 100)]
regs = players[players["is_regular"]]

replacement = pool.groupby("pos_group").apply(pooled_rates, include_groups=False)
regular     = regs.groupby("pos_group").apply(pooled_rates, include_groups=False)

comparison = pd.concat({"replacement": replacement, "regular": regular}).round(3)
print("Replacement pool size:", pool.groupby("pos_group").size().to_dict())
comparison

Replacement pool size: {'D': 46, 'F': 95}


goals_60  a1_60  ixg_60  xgf_60  xga_60
            pos_group                                         
replacement D             0.129  0.226   0.145   2.404   2.478
            F             0.529  0.359   0.666   2.228   2.382
regular     D             0.211  0.340   0.187   2.496   2.500
            F             0.710  0.574   0.714   2.524   2.499

## 8. Save
Saves the cleaned table to `data/skaters_clean.csv` for the next notebook. The `.gitignore` already keeps it off GitHub, since it's derived from MoneyPuck's data.

In [9]:
players.to_csv("../data/skaters_clean.csv", index=False)
replacement.to_csv("../data/replacement_rates.csv")
print("Saved", len(players), "players.")

Saved 940 players.


## Next: `02_war_model.ipynb`
- Turn each component (EV offense, EV defense, power play, penalty kill, penalties, faceoffs) into **goals above replacement**
- Add them up and convert goals to **wins**
- Check the leaderboard against public WAR models